# Trainer Addendum / Spark Performance Cheat Sheet Lab

This compact addendum closes the remaining gaps between:

1. the original training requirements,
2. the completed Examples 1–50 series, and
3. the Spark Performance Tuning cheat sheet provided for the training.

## Trainer objective

The objective remains:

> **Teach participants to diagnose and optimize Spark workloads rather than simply increasing cluster resources.**

The lab is organized as compact mini demonstrations rather than another large example series.

## Mini demonstrations

1. `repartition()` vs `coalesce()`
2. File size and the small-file problem
3. Data locality
4. Spark UI masterclass: Jobs → Stages → Tasks → Executors → SQL
5. GC, memory, spill, stragglers, and bottleneck diagnosis
6. Catalyst, Tungsten, and Whole-Stage Code Generation
7. AQE configuration cheat sheet lab
8. Dynamic partition overwrite
9. `REBALANCE` hint and write optimization
10. Storage-partitioned joins and modern table-layout concepts
11. Final integrated slow-job diagnosis exercise

---

## Important runtime note

The included data is intentionally tiny.

That is ideal for teaching APIs and execution plans, but it cannot reliably reproduce:

- production-sized spills,
- major GC pressure,
- severe skew,
- executor locality effects,
- AQE skew splitting,
- dynamic partition pruning in every Spark version,
- storage-partitioned join optimization.

For those topics, the notebook gives the **mechanics, plan inspection points, and trainer discussion workflow**. Use a larger cluster/data set if you want the Spark UI to visibly demonstrate the production symptom.

## Source mapping: gaps this addendum closes

### Original training requirements

- `coalesce()`
- file sizes
- small-file problem
- data locality
- Spark UI walkthrough
- Jobs / Stages / Tasks / Executors / SQL tab
- execution metrics
- stragglers
- GC and memory issues
- Tungsten
- practical troubleshooting

### Cheat-sheet-specific items

- `spark.sql.adaptive.coalescePartitions.parallelismFirst`
- `spark.sql.adaptive.advisoryPartitionSizeInBytes`
- `spark.sql.adaptive.coalescePartitions.initialPartitionNum`
- `spark.sql.sources.partitionOverwriteMode=dynamic`
- broadcast hint
- `spark.sql.adaptive.skewJoin.skewedPartitionFactor`
- `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes`
- `spark.sql.adaptive.forceOptimizeSkewedJoin`
- storage-partitioned join concept
- `REBALANCE` hint

In [ ]:
from pathlib import Path
import shutil
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

BASE_PATH = Path.cwd()
DATA_PATH = BASE_PATH / "data"

if not DATA_PATH.exists():
    candidate = Path("/mnt/data/spark_trainer_addendum_cheat_sheet_lab/data")
    if candidate.exists():
        DATA_PATH = candidate

assert DATA_PATH.exists(), "Could not find data/. Update BASE_PATH."

WORK_PATH = BASE_PATH / "work"
if str(BASE_PATH).startswith("/mnt/data/spark_trainer_addendum_cheat_sheet_lab"):
    WORK_PATH = BASE_PATH / "work"

if WORK_PATH.exists():
    shutil.rmtree(WORK_PATH)
WORK_PATH.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("Trainer-Addendum-Cheat-Sheet-Lab")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.warehouse.dir", str(WORK_PATH / "warehouse"))
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("segment", StringType(), True),
])

products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("brand", StringType(), True),
])

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_date", StringType(), True),
    StructField("region", StringType(), True),
    StructField("order_year", StringType(), True),
    StructField("order_month", StringType(), True),
])

customers_df = spark.read.option("header", True).schema(customers_schema).csv(str(DATA_PATH / "customers.csv"))
products_df = spark.read.option("header", True).schema(products_schema).csv(str(DATA_PATH / "products.csv"))
orders_df = spark.read.option("header", True).schema(orders_schema).csv(str(DATA_PATH / "orders.csv"))

print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

# Mini Demo 1 — `repartition()` vs `coalesce()`

## Training gap closed

- Repartitioning: already covered
- `coalesce()`: now explicitly demonstrated

## Mental model

```text
repartition()
    → can increase or decrease partitions
    → redistributes data
    → usually causes a shuffle

coalesce()
    → reduces partitions
    → tries to avoid a full shuffle
    → can create uneven partition sizes
```

## Trainer message

Do not choose based only on "which is faster."

Ask:

> **Do I need a full redistribution, or am I simply reducing parallelism?**

In [ ]:
print("Original partitions:", orders_df.rdd.getNumPartitions())

repartitioned_df = orders_df.repartition(8)
coalesced_df = repartitioned_df.coalesce(3)

print("After repartition(8):", repartitioned_df.rdd.getNumPartitions())
print("After coalesce(3):", coalesced_df.rdd.getNumPartitions())

print("\nREPARTITION PLAN")
repartitioned_df.explain("formatted")

print("\nCOALESCE PLAN")
coalesced_df.explain("formatted")

print("\nTeaching exercise:")
print("1. Ask learners which operation is more likely to redistribute data globally.")
print("2. Ask why coalesce is commonly useful before reducing output file counts.")
print("3. Ask why coalesce is not a substitute for balancing heavily skewed data.")

# Mini Demo 2 — File Size and the Small-File Problem

## Training gap closed

- File sizes
- Small-file problem

## The problem

```text
Too many tiny files
        ↓
Too many file listings / opens
        ↓
Too many input splits / tasks
        ↓
Scheduler overhead
        ↓
Poor cluster efficiency
```

Partition count and output file count are closely related, but they are not the same thing in every workload.

The right file size depends on:

- storage system,
- compression,
- cluster parallelism,
- downstream read patterns,
- data format,
- and workload scale.

This lab demonstrates the mechanics using deliberately small data.

In [ ]:
many_files_path = WORK_PATH / "many_small_files"
fewer_files_path = WORK_PATH / "fewer_files"

orders_df.repartition(8).write.mode("overwrite").parquet(str(many_files_path))
orders_df.repartition(2).write.mode("overwrite").parquet(str(fewer_files_path))

def parquet_files(path):
    return sorted(
        p for p in Path(path).rglob("*.parquet")
        if p.is_file()
    )

many_files = parquet_files(many_files_path)
fewer_files = parquet_files(fewer_files_path)

print("Many-file write:")
print("  data files:", len(many_files))
print("  total bytes:", sum(p.stat().st_size for p in many_files))

print("\nFewer-file write:")
print("  data files:", len(fewer_files))
print("  total bytes:", sum(p.stat().st_size for p in fewer_files))

print("\nTrainer note:")
print("The absolute sizes are tiny because this is a tiny dataset.")
print("Use this to teach the relationship between output partitioning and file counts.")
print("Do not infer a production file-size target from this toy run.")

# Mini Demo 3 — Data Locality

## Training gap closed

Data locality is difficult to reproduce meaningfully in `local[*]` mode because the notebook does not have distributed executors reading distributed storage blocks.

Therefore this is a **conceptual demonstration grounded in the training objective**.

## Mental model

```text
Best case:
Task runs where required data is already local
        ↓
Less network transfer

Less ideal:
Task must fetch significant data remotely
        ↓
More network dependency
```

In modern cloud storage architectures, locality behaves differently from classic HDFS block locality. The broader lesson remains:

> **Minimize unnecessary data movement.**

This is the same performance principle behind:

- partition pruning,
- broadcast joins,
- reducing shuffles,
- storage-aware layouts,
- and efficient table design.

In [ ]:
print("Data locality cannot be faithfully demonstrated in local[*] mode.")
print()
print("Trainer whiteboard prompt:")
print("Read → local/remote data movement → transform → shuffle → downstream task")
print()
print("Ask learners:")
print("1. Which operations preserve existing partition placement?")
print("2. Which operations force redistribution?")
print("3. Why is shuffle often more important to optimize than adding CPU?")
print("4. How do partition pruning and broadcast joins reduce unnecessary movement?")

# Mini Demo 4 — Spark UI Masterclass

## Training gap closed

- Jobs
- Stages
- Tasks
- Executors
- SQL tab
- execution metrics

## Use this deliberately shaped query

Run the query below, then open the Spark UI.

Trainer walkthrough order:

```text
Application
    ↓
Jobs
    ↓
Stages
    ↓
Tasks
    ↓
Executors
    ↓
SQL / query details
```

### Questions learners should answer

1. Which action created the job?
2. How many stages were created?
3. Where did stage boundaries come from?
4. Which stages involve shuffle?
5. Are task durations balanced?
6. Is there significant shuffle read/write?
7. Is there spill?
8. Is GC time unusually high?
9. What does the SQL physical plan predict before execution?

In [ ]:
ui_demo_df = (
    orders_df
    .join(products_df, "product_id")
    .filter(F.col("amount") >= 300)
    .groupBy("region", "category")
    .agg(
        F.sum("amount").alias("revenue"),
        F.count("*").alias("orders")
    )
)

print("PHYSICAL PLAN BEFORE ACTION")
ui_demo_df.explain("formatted")

print("\nTriggering an action. Inspect the Spark UI after this completes.")
ui_demo_df.collect()

print("\nSpark UI trainer checklist:")
print("- Jobs: overall action and duration")
print("- Stages: shuffle boundaries and time distribution")
print("- Tasks: stragglers and task-duration imbalance")
print("- Executors: GC time, memory, task distribution")
print("- SQL: operator-level query details and plan")

# Mini Demo 5 — GC, Memory, Spill, Stragglers, and Bottleneck Diagnosis

## Training gap closed

- GC and memory issues
- Spill
- Stragglers
- Identifying bottlenecks

A tiny local dataset will not reliably create meaningful production GC pressure or disk spill.

Instead, this mini lab teaches the diagnostic signatures.

## Diagnostic patterns

### Skew / straggler

```text
Most tasks finish quickly
One or a few tasks run much longer
```

### Spill

```text
Large intermediate data
Memory pressure
Disk spill metrics increase
```

### GC pressure

```text
High GC time
Reduced useful CPU time
Possible memory churn
```

The key lesson:

> **Do not tune memory first. Identify whether the root cause is skew, shuffle volume, oversized partitions, bad joins, or actual memory pressure.**

In [ ]:
print("Potential skew key distribution:")
orders_df.groupBy("customer_id").count().orderBy(F.desc("count")).show()

diagnostic_df = (
    orders_df
    .repartition(8, "customer_id")
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("customer_revenue"))
)

print("\nDIAGNOSTIC PLAN")
diagnostic_df.explain("formatted")

print("\nTriggering action for Spark UI inspection.")
diagnostic_df.collect()

print("\nProduction UI interpretation:")
print("- One long task: investigate skew / straggler.")
print("- Large shuffle read/write: investigate partitioning and joins.")
print("- Spill: inspect intermediate data size and memory pressure.")
print("- High GC time: investigate memory churn and oversized workloads.")
print("- Do not treat 'add more executors' as the first diagnosis.")

# Mini Demo 6 — Catalyst, Tungsten, and Whole-Stage Code Generation

## Training gap closed

- Catalyst Optimizer: already covered, now summarized
- Tungsten: explicitly introduced
- Whole-Stage Code Generation: explicitly introduced

## Mental model

```text
User DataFrame / SQL
        ↓
Catalyst
Logical analysis and optimization
        ↓
Physical plan selection
        ↓
Tungsten-oriented execution optimizations
        ↓
Whole-stage code generation where applicable
        ↓
Execution
```

For this training, keep the distinction simple:

- **Catalyst:** optimizes and transforms query plans.
- **Tungsten:** Spark's execution-engine effort around efficient CPU and memory use.
- **Whole-stage code generation:** combines compatible operators into generated execution code where supported.

Use `explain("codegen")` as an inspection tool. Exact output varies by Spark version and query.

In [ ]:
codegen_df = (
    orders_df
    .filter(F.col("amount") > 300)
    .select(
        "customer_id",
        "product_id",
        (F.col("amount") * F.lit(1.1)).alias("adjusted_amount")
    )
)

print("FORMATTED PLAN")
codegen_df.explain("formatted")

print("\nCODEGEN OUTPUT")
codegen_df.explain("codegen")

print("\nTrainer prompt:")
print("Ask learners to distinguish:")
print("- logical optimization")
print("- physical operators")
print("- generated execution code")
print()
print("Avoid teaching Tungsten as a configuration knob.")
print("Teach it as part of why built-in Spark expressions generally integrate better with the execution engine than opaque Python UDFs.")

# Mini Demo 7 — AQE Configuration Cheat Sheet Lab

This section explicitly covers the missing cheat-sheet knobs.

## 1. Parallelism First

```text
spark.sql.adaptive.coalescePartitions.parallelismFirst
```

The cheat sheet recommends `false`.

Training interpretation:

- `true`: AQE favors preserving more parallelism.
- `false`: AQE more strongly follows advisory partition sizing.

## 2. Advisory Partition Size

```text
spark.sql.adaptive.advisoryPartitionSizeInBytes
```

The cheat sheet recommends `128–256 MB`.

This is a target used by adaptive logic; it is not a universal guarantee that every output partition will be exactly that size.

## 3. Initial Partition Count

```text
spark.sql.adaptive.coalescePartitions.initialPartitionNum
```

This controls the initial partition count considered by adaptive coalescing.

## 6–8. Skew Join Settings

```text
spark.sql.adaptive.skewJoin.skewedPartitionFactor
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes
spark.sql.adaptive.forceOptimizeSkewedJoin
```

Use these only after identifying a real skew problem.

The training sequence remains:

> **Observe skew → inspect plan/metrics → understand thresholds → tune only if necessary.**

In [ ]:
# Save existing values so the lab does not permanently change the session.
aqe_keys = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    "spark.sql.adaptive.coalescePartitions.initialPartitionNum",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.adaptive.forceOptimizeSkewedJoin",
]

original_aqe = {}
for key in aqe_keys:
    try:
        original_aqe[key] = spark.conf.get(key)
    except Exception:
        original_aqe[key] = None

# Use strings because Spark conf values are represented as strings.
demo_values = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes": str(128 * 1024 * 1024),
    "spark.sql.adaptive.coalescePartitions.initialPartitionNum": "8",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor": "3",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes": str(256 * 1024 * 1024),
}

for key, value in demo_values.items():
    try:
        spark.conf.set(key, value)
    except Exception as exc:
        print(f"Could not set {key}: {exc}")

# This setting may not exist in older Spark versions.
try:
    spark.conf.set("spark.sql.adaptive.forceOptimizeSkewedJoin", "true")
except Exception as exc:
    print("forceOptimizeSkewedJoin is not available in this Spark runtime:", exc)

print("AQE CHEAT-SHEET SETTINGS NOW VISIBLE TO THIS SESSION:")
for key in aqe_keys:
    try:
        print(f"{key} = {spark.conf.get(key)}")
    except Exception as exc:
        print(f"{key} = <not available: {exc}>")

aqe_demo_df = (
    orders_df
    .repartition(8, "customer_id")
    .join(products_df, "product_id")
    .groupBy("region")
    .agg(F.sum("amount").alias("revenue"))
)

print("\nAQE DEMO PLAN")
aqe_demo_df.explain("formatted")
aqe_demo_df.collect()

print("\nTrainer warning:")
print("These values are lab examples and cheat-sheet defaults/recommendations, not universal production settings.")
print("Tune against actual data distribution and Spark UI metrics.")

# Restore original settings where possible.
for key, value in original_aqe.items():
    if value is not None:
        try:
            spark.conf.set(key, value)
        except Exception:
            pass

# Mini Demo 8 — Dynamic Partition Overwrite

## Training gap closed

Cheat-sheet item:

```text
spark.sql.sources.partitionOverwriteMode = dynamic
```

## Why it matters

For partitioned writes:

```text
static overwrite
    → overwrite behavior can affect a broader partition scope

dynamic overwrite
    → only partitions represented by incoming data are replaced
```

The exact behavior should always be validated for your Spark version and table format.

This example uses a partitioned Parquet path to make the mechanics visible.

In [ ]:
overwrite_path = WORK_PATH / "dynamic_partition_overwrite"

original_overwrite_mode = spark.conf.get("spark.sql.sources.partitionOverwriteMode")

try:
    # Initial write creates multiple region partitions.
    (
        orders_df
        .write
        .mode("overwrite")
        .partitionBy("region")
        .parquet(str(overwrite_path))
    )

    print("Initial partitions:")
    for p in sorted(Path(overwrite_path).glob("region=*")):
        print(" ", p.name)

    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

    # Incoming data contains only East.
    east_replacement_df = (
        orders_df
        .filter(F.col("region") == "East")
        .withColumn("amount", F.col("amount") + 10)
    )

    (
        east_replacement_df
        .write
        .mode("overwrite")
        .partitionBy("region")
        .parquet(str(overwrite_path))
    )

    print("\nPartitions after dynamic overwrite:")
    for p in sorted(Path(overwrite_path).glob("region=*")):
        print(" ", p.name)

    print("\nRead-back result:")
    spark.read.parquet(str(overwrite_path)).groupBy("region").count().show()

finally:
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", original_overwrite_mode)

# Mini Demo 9 — `REBALANCE` Hint and Write Optimization

## Training gap closed

Cheat-sheet item:

```text
/*+ REBALANCE(...) */
```

The goal is to improve partition balance before a downstream operation such as a write.

This concept connects:

- partition sizing,
- file counts,
- AQE,
- and output layout.

Unlike a blanket `repartition()`, a rebalance hint expresses the intent to rebalance data for downstream processing.

Exact plan behavior depends on Spark version and AQE support.

In [ ]:
orders_df.createOrReplaceTempView("orders_for_rebalance")

rebalance_df = spark.sql(
    """
    SELECT /*+ REBALANCE(region) */
           order_id,
           customer_id,
           product_id,
           amount,
           region
    FROM orders_for_rebalance
    """
)

print("REBALANCE PLAN")
rebalance_df.explain("formatted")

print("\nRESULT")
rebalance_df.show()

print("\nTrainer discussion:")
print("- REBALANCE is about improving downstream partition balance.")
print("- It is especially relevant before writes when uneven output partition sizes are a concern.")
print("- Inspect the actual plan; hint behavior depends on Spark/runtime support.")

# Mini Demo 10 — Storage-Partitioned Joins and Modern Table Layouts

## Training gap closed

Cheat-sheet item:

> Storage-partitioned join

This is intentionally a **conceptual advanced section** because the small CSV/Parquet lab does not provide a realistic modern DataSource V2 table environment.

The idea is:

```text
Storage layout contains useful partitioning / distribution information
        ↓
Both sides of a join can potentially exploit compatible layout
        ↓
Spark may avoid some otherwise unnecessary shuffle work
```

This should be distinguished from, but connected to:

- classic bucketing,
- partition-aware joins,
- table clustering/distribution,
- DataSource V2 capabilities.

The exact optimization depends on:

- Spark version,
- table format,
- connector,
- catalog,
- source capabilities,
- and compatible partition/distribution information.

Therefore this lab does **not** claim that ordinary Parquet automatically gives you a storage-partitioned join optimization.

In [ ]:
# We can still demonstrate the prerequisite idea: aligned layout and join keys.
left_path = WORK_PATH / "layout_left"
right_path = WORK_PATH / "layout_right"

(
    orders_df
    .select("customer_id", "amount", "region")
    .write
    .mode("overwrite")
    .partitionBy("region")
    .parquet(str(left_path))
)

(
    customers_df
    .select("customer_id", "city", "segment")
    .write
    .mode("overwrite")
    .parquet(str(right_path))
)

left_layout_df = spark.read.parquet(str(left_path))
right_layout_df = spark.read.parquet(str(right_path))

storage_join_candidate_df = (
    left_layout_df
    .join(right_layout_df, "customer_id")
)

print("STORAGE-LAYOUT DISCUSSION PLAN")
storage_join_candidate_df.explain("formatted")

print("\nTrainer message:")
print("This plan is NOT proof of storage-partitioned join optimization.")
print("Use a compatible modern table source/runtime to demonstrate the actual feature.")
print("The purpose here is to connect table layout to potential shuffle avoidance.")

# Mini Demo 11 — Final Integrated Slow-Job Diagnosis Exercise

This is the recommended closure exercise.

## Scenario

Participants are told:

> "The Spark job is slow. The team wants to add more executors."

Their job is to prove or disprove that adding resources is the correct first action.

## Workflow

```text
Read
  ↓
Inspect data
  ↓
Inspect key distribution
  ↓
Inspect plan
  ↓
Run job
  ↓
Open Spark UI
  ↓
Identify bottleneck
  ↓
Apply one targeted optimization
  ↓
Compare result + plan
```

The tiny dataset will not create a dramatic slow job, but the workflow is exactly the one participants should apply to real workloads.

In [ ]:
# Intentionally baseline-style query.
slow_job_baseline_df = (
    orders_df
    .join(customers_df, "customer_id")
    .join(products_df, "product_id")
    .filter(F.col("amount") >= 300)
    .groupBy("customer_id", "region", "category")
    .agg(
        F.sum("amount").alias("revenue"),
        F.count("*").alias("order_count")
    )
)

print("STEP 1 — BASELINE DATA DISTRIBUTION")
orders_df.groupBy("customer_id").count().orderBy(F.desc("count")).show()

print("\nSTEP 2 — BASELINE PLAN")
slow_job_baseline_df.explain("formatted")

print("\nSTEP 3 — EXECUTE BASELINE")
baseline_result = sorted(slow_job_baseline_df.collect())

# Targeted optimization: early projection/filter + broadcast small dimensions.
optimized_slow_job_df = (
    orders_df
    .filter(F.col("amount") >= 300)
    .select("customer_id", "product_id", "region", "amount")
    .join(
        F.broadcast(customers_df.select("customer_id")),
        "customer_id"
    )
    .join(
        F.broadcast(products_df.select("product_id", "category")),
        "product_id"
    )
    .groupBy("customer_id", "region", "category")
    .agg(
        F.sum("amount").alias("revenue"),
        F.count("*").alias("order_count")
    )
)

print("\nSTEP 4 — OPTIMIZED PLAN")
optimized_slow_job_df.explain("formatted")

print("\nSTEP 5 — EXECUTE OPTIMIZED VERSION")
optimized_result = sorted(optimized_slow_job_df.collect())

print("\nSTEP 6 — CORRECTNESS VALIDATION")
print("Results identical:", baseline_result == optimized_result)

print("\nFinal trainer questions:")
print("1. What evidence identified the bottleneck?")
print("2. Did we reduce data movement?")
print("3. Did we change the join strategy?")
print("4. Did we change partitioning unnecessarily?")
print("5. What would you check next in the Spark UI?")
print("6. Only now: is additional cluster capacity actually justified?")

# Final Trainer Cheat Sheet

## Diagnose before tuning

### If the job is slow

```text
1. What action triggered the job?
2. Which stage is slow?
3. Are tasks balanced?
4. Is there shuffle?
5. Is there skew?
6. Is there spill?
7. Is GC high?
8. What does the physical plan show?
9. Can we reduce data movement?
10. Only then consider more resources.
```

---

## Symptom → likely investigation

| Symptom | Investigate |
|---|---|
| Many tiny tasks/files | Small-file problem, partition/file sizing |
| Too few huge tasks | Partition size, skew, insufficient parallelism |
| Large shuffle | Join strategy, repartitioning, aggregation |
| One long task | Skew / straggler |
| High spill | Intermediate data size, memory pressure, partition sizing |
| High GC | Memory churn, large objects, oversized tasks |
| Large join cost | Broadcast eligibility, sort-merge behavior, data layout |
| Repeated scan | Caching/persistence only when justified |
| Expensive scan | Predicate pushdown, projection pruning, partition pruning |
| Slow write | Output partitioning, `REBALANCE`, file count |
| Adaptive plan surprises | AQE configuration + final executed plan |

---

# The course closure

> **Read → Transform → Shuffle → Diagnose → Optimize**

And the discipline behind it:

> **Do not increase cluster resources until you understand where the existing resources are being spent.**

This addendum is designed to be delivered after Examples 1–50 as a compact trainer-led lab and final consolidation exercise.